In [2]:
import pandas as pd
import re
import numpy as np

In [3]:
# Path to the folder where you downloaded the files
folder_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/association_overall_direct'

# Read all parquet parts at once
df = pd.read_parquet(folder_path, engine='pyarrow')

# Show the first few rows
print(df.head(3))

      diseaseId         targetId     score  evidenceCount
0  DOID_0050890  ENSG00000001084  0.031799              4
1  DOID_0050890  ENSG00000004142  0.002217              1
2  DOID_0050890  ENSG00000004478  0.002217              1


In [16]:
df.columns

Index(['diseaseId', 'targetId', 'score', 'evidenceCount'], dtype='object')

In [4]:
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/disgent_with_time.csv')
ppi_feature = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/ppi_full_2016_700_emb.csv')
all_df = all_df[all_df['string_id'].isin(ppi_feature['string_id'])]
time = 2017
selected_diseases = []
for disease_id in all_df['disease_id'].unique():
    sub_df = all_df[all_df['disease_id']==disease_id]
    if len(sub_df) < 15:
        continue
    else:
        # print(type(time),type(sub_df['first_pub_year'].max()))
        if sub_df['first_pub_year'].max() > time and sub_df['first_pub_year'].min() <= time and len(sub_df[sub_df['first_pub_year']<time]) >=5:
            selected_diseases.append(disease_id)

In [5]:
icd2do_map = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ICD2DO.tsv',sep='\t')
icd2do_map.head(3)

,id,label,xrefs
0,DOID:0014667,disease of metabolism,ICD10CM:E88.9
1,DOID:0040008,isoniazide allergy,ICD10CM:Z88.1
2,DOID:0040010,mepivacaine allergy,ICD10CM:Z88.4


In [9]:
icd2do_map['icd'] = icd2do_map['xrefs'].str.split(':').str[1].str.split('.').str[0]

In [78]:
query_disease_map = icd2do_map[icd2do_map['icd'].isin([items[-3: ]for items in selected_diseases])]

In [79]:
query_disease_map = query_disease_map.copy()
query_disease_map['diseaseId'] = query_disease_map['id'].str.replace(':', '_', regex=False)

In [80]:
query_disease_map = query_disease_map.drop(columns=['id'])

In [38]:
ot_disease = pd.read_parquet('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/disease/disease.parquet')
ot_disease.head(2)

,id,code,name,description,dbXRefs,parents,synonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology
0,DOID_0050890,http://purl.obolibrary.org/obo/DOID_0050890,synucleinopathy,A neurodegenerative disease that is characteri...,"[MESH:D000080874, MONDO:0000510, UMLS:C5191670...","[MONDO_0019052, MONDO_0021179, MONDO_0024237]",{'hasExactSynonym': ['alpha Synucleinopathies'...,[],[],"[EFO_0006792, EFO_1001050]","[MONDO_0024237, EFO_0005772, EFO_0000618, MOND...","[EFO_0000618, OTAR_0000018, OTAR_0000020]","[MONDO_0000211, MONDO_0016418, MONDO_0014889, ...","{'isTherapeuticArea': False, 'leaf': False, 's..."
1,DOID_10113,http://purl.obolibrary.org/obo/DOID_10113,trypanosomiasis,Infection with protozoa of the genus trypanosoma.,"[UMLS:C0041227, MONDO:0000940, ICD10CM:B56, Me...",[MONDO_0002428],{'hasExactSynonym': ['Trypanosoma caused disea...,[],[],"[MONDO_0001444, EFO_0005225, EFO_0008559]","[MONDO_0002428, EFO_0001067, EFO_0005741]",[EFO_0005741],"[EFO_0005225, EFO_0005529, EFO_0008559, MONDO_...","{'isTherapeuticArea': False, 'leaf': False, 's..."


In [81]:
query_disease_map = query_disease_map.rename(columns={'label': 'name'})
query_disease_direct_name_map = pd.merge(query_disease_map, ot_disease, on='name', how='inner')

In [82]:
########## directly mapped disease
query_disease_direct_name_map['icd'].unique().shape

(38,)

In [83]:
missing_disease = list(set([items[-3: ]for items in selected_diseases]) - set(query_disease_direct_name_map['icd'].unique().tolist()))

In [84]:
set(missing_disease) - set(icd2do_map[icd2do_map['icd'].isin(missing_disease)]['icd'].unique().tolist())
######## note: we should delete F90

{'F90'}

In [85]:
near_disease_map = {
    'G20': ['Parkinsonism', 'Parkinson disease', 'late-onset Parkinson disease', 'young-onset Parkinson disease','hemiparkinsonism-hemiatrophy syndrome','Parkinson disease, dominant','Hereditary late-onset Parkinson disease'],
    'D57': ['sickle cell anemia',
            'sickle cell-beta-thalassemia disease syndrome',
            'sickle cell-hemoglobin c disease syndrome',
            'sickle cell-hemoglobin d disease syndrome',
            'sickle cell-hemoglobin E disease syndrome',
            'hereditary persistence of fetal hemoglobin-sickle cell disease syndrome',
            'sickle cell disease and related diseases',
            'Sickle cell - beta-thalassemia disease',
            'Sickle cell - hemoglobin C disease',
            'Sickle cell - hemoglobin D disease',
            'Sickle cell - hemoglobin E disease'],
    'G10': ['Huntington disease','juvenile Huntington disease'],
    'G30': ['Alzheimer disease',
            'Alzheimer disease type 1',
            'Alzheimer disease 3',
            'Alzheimer disease 18',
            'early-onset autosomal dominant Alzheimer disease',
            'familial Alzheimer disease',
            'late-onset Alzheimers disease'],
    'J80': ['adult acute respiratory distress syndrome','acute respiratory distress syndrome'],
    'L80': ['Vitiligo'],
    'N97': ['anovulation'],
    'L20': ['recalcitrant atopic dermatitis'],
    'C81': ['classic Hodgkin lymphoma',
            'Hodgkins lymphoma',
            'nodular sclerosis Hodgkin lymphoma',
            'Splenic Hodgkin Lymphoma',
            'Hodgkins lymphoma, mixed cellularity'],
    'C43': ['cutaneous melanoma',
            'amelanotic skin melanoma',
            'amelanotic melanoma',
            'superficial spreading melanoma',
            'lentigo maligna melanoma',
            'nodular melanoma',
            'desmoplastic melanoma',
            'spindle cell melanoma',
            'childhood malignant melanoma',
            'melanoma',
            'metastatic melanoma',
            'melanoma, cutaneous malignant, susceptibility to, 1',
            'melanoma, cutaneous malignant, susceptibility to, 2',
            'melanoma, cutaneous malignant, susceptibility to, 3',
            'melanoma, cutaneous malignant, susceptibility to, 8',
            'melanoma, cutaneous malignant, susceptibility to, 9',
            'susceptibility to familial cutaneous melanoma',
            'familial melanoma',
            'familial atypical multiple mole melanoma syndrome']
}

In [88]:
query_disease_missing = query_disease_map[query_disease_map['icd'].isin(missing_disease)]

In [93]:
# Make a copy to avoid SettingWithCopyWarning
query_disease_missing = query_disease_missing.copy()

# Expand rows
expanded = query_disease_missing.explode('icd').apply(
    lambda row: pd.DataFrame({
        **{col: [row[col]] * len(near_disease_map.get(row['icd'], [])) for col in query_disease_missing.columns},
        'name': near_disease_map.get(row['icd'], [])
    }),
    axis=1
)

# Concatenate the resulting small DataFrames
query_disease_expanded = pd.concat(expanded.tolist(), ignore_index=True)

In [ ]:
query_disease_missing = pd.merge(query_disease_expanded, ot_disease, on='name', how='inner')
query_disease_missing['icd'].unique().shape

(10,)

In [98]:
query_disease_clean_map = pd.concat([query_disease_missing, query_disease_direct_name_map], ignore_index=True)

In [99]:
query_disease_clean_map['icd'].unique().shape

(48,)

In [103]:
query_dga = pd.merge(query_disease_clean_map[['id','icd']].rename(columns={'id': 'diseaseId'}), df, on='diseaseId', how='inner')

In [106]:
all_df.head(2)

,disease_id,omim,hpo,disease_name,gene_id,score,first_pub_year,last_pub_year,ei,dsi,dpi,uniprot_id,string_id,ori_annotation
0,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,ERBB2,1.0,2004.0,2007.0,0.917,0.298,0.957,P04626,9606.ENSP00000269571,True
1,ICD10_C16,OMIM_613659,HPO_HP:0012126,Malignant neoplasm of stomach,PIK3CA,1.0,2004.0,2023.0,0.978,0.275,0.957,P42336,9606.ENSP00000263967,True


In [ ]:
query_dga['disease_id'] = 'ICD10_'+query_dga['icd']

In [105]:
import mygene

In [112]:
mg = mygene.MyGeneInfo()
# Query mygene for UniProt and Entrez gene ID mappings
results = mg.querymany(
    query_dga['targetId'].unique().tolist(),
    scopes='ensembl.gene',
    fields='ensembl.protein',
    species='human'
)
results_df = pd.DataFrame(results)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


5 input query terms found dup hits:	[('ENSG00000282304', 2), ('ENSG00000249738', 2), ('ENSG00000188660', 2), ('ENSG00000280018', 2), ('E
7 input query terms found no hit:	['ENSG00000168078', 'ENSG00000189144', 'ENSG00000291190', 'ENSG00000281376', 'ENSG00000310560', 'ENS


In [113]:
results_df = results_df[~results_df['ensembl'].isna()]

In [115]:
map_df = pd.DataFrame({
    'ensg': results_df['query'],
    'ensp': results_df['ensembl'].apply(lambda x: x.get('protein') if isinstance(x, dict) else None)
})

# Step 2: Explode the list of proteins to one per row
map_df = map_df.explode('ensp').reset_index(drop=True)

In [117]:
query_dga = query_dga.rename(columns={'targetId': 'ensg'})
mapped_ot_dga = pd.merge(query_dga, map_df, on='ensg', how='left')

In [118]:
mapped_ot_dga['string_id'] = '9606.'+mapped_ot_dga['ensp']

In [119]:
# Merge with indicator to find rows only in df2
merged = mapped_ot_dga.merge(all_df[['disease_id', 'string_id']], 
                   on=['disease_id', 'string_id'], 
                   how='left', 
                   indicator=True)

# Keep only rows that are not in df1
query_dga_filtered = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

In [122]:
query_dga_filtered = query_dga_filtered[['score', 'disease_id', 'string_id']]

In [123]:
len(query_dga_filtered)

2561123

In [124]:
query_dga_filtered = query_dga_filtered.drop_duplicates(subset=['disease_id', 'string_id'])
len(query_dga_filtered)

1095394

In [125]:
query_dga_filtered['disease_id'].unique().shape

(47,)

In [128]:
query_dga_filtered = query_dga_filtered.round(4)

In [130]:
query_dga_filtered[['score', 'disease_id', 'string_id']].to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/opentarget/ot_dgas_2017_47_delF90.csv',index=False)